# Advanced RAG (Parent-Document · RAG-Fusion · Self-RAG) — built from scratch

Naive RAG plateaus for three structural reasons; this notebook builds the three fixes and
**measures** each on the **real** `all-MiniLM-L6-v2` encoder over chapter 2's Helios-7 document:

1. **Parent-Document** — retrieve on SMALL children (sharp), feed the LLM the larger PARENT (context).
2. **RAG-Fusion** — multi-query + RRF (chapter 7's machinery), recovering facts a single probe misses.
3. **Self-RAG** — a retrieve-on-demand GATE and a SUPPORT check (ground before you trust).

> **Honesty (read first):** parent-document retrieval, RAG-Fusion, and the Self-RAG **support check**
> are fully implementable and **measured** here on the real encoder. Self-RAG's true reflection tokens
> (`ISREL`/`ISSUP`/`ISUSE`) need a specially **trained** LLM — this env is encoder-only, so we do NOT
> fake them or the generated answer text. We implement the two decisions that ARE computable with an
> encoder (the gate and the support check) and label the generator/critique *text* as illustrative.
> **Real:** encoder, all retrieval, the support-check math. **Illustrative:** generated answer/critique
> text and the gate's hand-coded rule. Every number below is measured, asserted before it is claimed.

## Step 1 — set up: import the canonical functions

Everything uses the functions in `advanced_rag.py` (which reuse chapter 5's `DenseRetriever` + RRF +
metrics and chapter 7's `retrieve_multiquery`). We print versions + device for reproducibility; the
encoder is CPU-pinned (deterministic), and we report the available accelerator only for context.

In [1]:
import numpy as np

from advanced_rag import (
    SUPPORT_THRESHOLD,
    TOP_K,
    ParentDocumentRetriever,
    build_hierarchy,
    is_supported,
    needs_retrieval,
    recall_at_k,
    retrieve_multiquery,
    support_score,
)
from hybrid_search import RRF_K

print('numpy:', np.__version__)
try:
    import torch
    avail = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    print('torch:', torch.__version__, '| accelerator available:', avail, '| encoder runs on: cpu')
except ImportError:
    print('torch: not installed (retrieval is pure numpy — unaffected)')

parents, children = build_hierarchy()
retriever = ParentDocumentRetriever(parents, children)
print(f'document: {len(parents)} parent sections, {len(children)} child sentences')
print(f'dense lens: {retriever.backend}')

numpy: 2.4.6


torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


document: 3 parent sections, 6 child sentences
dense lens: sentence-transformers/all-MiniLM-L6-v2


## Step 2 — the child→parent hierarchy

Chapter 2's multi-section Helios-7 spec: each **section** is a PARENT (the context we feed the LLM),
each **sentence** a CHILD (the small unit we embed and retrieve on). Every child stores its parent id.

In [2]:
for pid, p in enumerate(parents):
    print(f'PARENT[{pid}]  # {p.heading}')
for ci, c in enumerate(children):
    print(f'  child[{ci}] (parent {c.parent_id}): {c.text}')

PARENT[0]  # Helios-7 Mission Overview
PARENT[1]  # Instruments
PARENT[2]  # Orbit
  child[0] (parent 0): The Helios-7 satellite is an Earth-observation platform operated by the Nairobi office.
  child[1] (parent 0): It was launched on March 3rd, 2024 from the Kourou spaceport aboard an Ariane 6 rocket.
  child[2] (parent 1): Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  child[3] (parent 1): The imager captures 200 spectral bands across the visible and near-infrared range.
  child[4] (parent 2): Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  child[5] (parent 2): This orbit keeps the local solar time of each pass roughly constant for stable imaging.


## Step 3 — the fragment problem: a sharp retrieval that hands over a context-starved child

Ask *"What keeps the local solar time of each pass roughly constant?"*. The top child is correct —
but it opens with **"This orbit"**, whose referent (`sun-synchronous orbit`) is in the *previous*
sentence, outside the child. The #1 child alone is a fragment; its parent section supplies the referent.

In [3]:
query = 'What keeps the local solar time of each pass roughly constant?'
result = retriever.retrieve(query, k=TOP_K)
top_child = children[result.child_indices[0]]
top_child_ctx = retriever.top_child_context(result)  # the #1 child alone (top-1 child-only RAG)
parent_ctx = retriever.parent_context(result)
print(f'query: {query}')
print(f'  top child: child[{result.child_indices[0]}] (cos={result.child_scores[0]:.3f})')
print(f'  child text: {top_child.text!r}')
referent = 'sun-synchronous orbit'
child_has = referent in top_child_ctx
parent_has = referent in parent_ctx
print(f"  referent {referent!r} in the #1 child alone: {child_has}")
print(f"  referent {referent!r} in its parent section: {parent_has}")
assert not child_has, 'the #1 child alone should LACK the referent (the fragment problem)'
assert parent_has, 'the parent section must SUPPLY the referent (retrieve small, read large)'
print('  -> the #1 child is a context-starved fragment; its parent supplies the referent.')

query: What keeps the local solar time of each pass roughly constant?
  top child: child[5] (cos=0.662)
  child text: 'This orbit keeps the local solar time of each pass roughly constant for stable imaging.'
  referent 'sun-synchronous orbit' in the #1 child alone: False
  referent 'sun-synchronous orbit' in its parent section: True
  -> the #1 child is a context-starved fragment; its parent supplies the referent.


![Animated — retrieve small, read large: the #1 child sentence lights up, then expands to reveal its
full parent section (the `sun-synchronous orbit` referent materializing around it). Child, parent, and
cosine are real all-MiniLM outputs. Generated by `make_animation_08.py`.](../../images/rag08_retrieve_small_read_large.gif)

## Step 4 — parent-document retrieval: precision × context, measured

The whole value proposition: sharp retrieval (the top child is in the correct section) AND more
context (the parent is larger than the child), at once. Both numbers are the encoder's own.

In [4]:
orbit_parent = next(i for i, p in enumerate(parents) if p.heading == 'Orbit')
top_child_parent = children[result.child_indices[0]].parent_id
print(f"  top child's parent section: {parents[top_child_parent].heading!r} (target: 'Orbit')")
assert top_child_parent == orbit_parent, 'the sharp child retrieval must hit the Orbit section'
child_chars = len(top_child.text)
parent_chars = len(parents[top_child_parent].text)
print(f'  child delivered to LLM: {child_chars} chars | parent delivered: {parent_chars} chars')
print(f'  context multiplier (parent/child): {parent_chars / child_chars:.1f}x more context, SAME sharp retrieval')
print(f'  deduped parents returned (unique, first-hit order): {list(result.parent_ids)}')
assert parent_chars > child_chars, 'the parent must carry more context than the child'

  top child's parent section: 'Orbit' (target: 'Orbit')
  child delivered to LLM: 87 chars | parent delivered: 179 chars
  context multiplier (parent/child): 2.1x more context, SAME sharp retrieval
  deduped parents returned (unique, first-hit order): [2, 1]


![The child→parent hierarchy and retrieve-small/read-large flow (left) and the measured precision ×
context (right): 87→179 chars (2.1×), same sharp 'Orbit' hit. Generated by
`make_figures_08.py`.](../../images/rag08_parent_child.png)

![Retrieve small, read large — measured: child-only 87 chars vs parent-document 179 chars (2.1× more
context), both hit the 'Orbit' section. Generated by `make_figures_08.py`.](../../images/rag08_precision_context.png)

## Step 5 — RAG-Fusion: recover a fact the single vague query misses (ch7 reused)

RAG-Fusion = multi-query + RRF (chapter 7), run over the CHILD index. A vague query misses the specific
"200 spectral bands" child at top-2; fusing three sharper reformulations recovers it (recall 0 → 1).

In [5]:
fusion_k = 2  # tight top-2 so recall is not trivially saturated on 6 children
vague = 'What are the imaging capabilities of Helios-7?'
reformulations = (
    'How many spectral bands does Helios-7 capture?',
    'What wavelength range does the imager cover?',
    'What kind of imager is on Helios-7?',
)
gold = next(i for i, c in enumerate(children) if 'spectral bands' in c.text)
raw = retriever._dense.search(vague, k=fusion_k).indices
fused = retrieve_multiquery(retriever._dense, (vague, *reformulations), k=fusion_k, k_rrf=RRF_K)
def rank(indices, g):
    return f'#{list(indices).index(g)+1}' if g in indices else 'MISS'
print(f'vague query: {vague}')
print(f'  RAW child hits (top-{fusion_k}) {list(raw)}   gold child[{gold}] rank: {rank(raw, gold)}')
print(f'  RAG-FUSION child hits (top-{fusion_k}) {list(fused)}   gold child[{gold}] rank: {rank(fused, gold)}')
print(f'  recall@{fusion_k}: raw {recall_at_k(raw, gold):.0f} -> RAG-Fusion {recall_at_k(fused, gold):.0f}')
assert recall_at_k(raw, gold) == 0.0, 'the vague single query should MISS the gold at top-2'
assert recall_at_k(fused, gold) == 1.0, 'RAG-Fusion must recover the gold the single query missed'

vague query: What are the imaging capabilities of Helios-7?
  RAW child hits (top-2) [2, 0]   gold child[3] rank: MISS
  RAG-FUSION child hits (top-2) [2, 3]   gold child[3] rank: #2
  recall@2: raw 0 -> RAG-Fusion 1


![RAG-Fusion recovers a fact the single vague query misses (top-2): raw [2,0] gold MISS, fused [2,3]
gold #2, recall 0→1. Generated by `make_figures_08.py`.](../../images/rag08_ragfusion_recovery.png)

## Step 6 — Self-RAG (a): the retrieve-on-demand gate

Naive RAG always retrieves. Self-RAG's `Retrieve` token decides *when* — skip retrieval when the query
needs no external fact. The rule here is an illustrative stand-in for the trained token; the *behaviour*
(don't retrieve when it can't help) is the real Self-RAG lesson.

In [6]:
gate_queries = [
    ('When was Helios-7 launched?', True),
    ('Hello, how are you today?', False),
    ('What is 2+2?', False),
    ('What is the ground resolution of the Helios-7 imager?', True),
]
print(f"  {'query':<52} | {'gate: retrieve?':>15} | expected")
print('  ' + '-' * 84)
for q, expected in gate_queries:
    decision = needs_retrieval(q)
    print(f'  {q:<52} | {str(decision):>15} | {expected}')
    assert decision == expected, f'gate decision wrong for {q!r}'
print('  -> the gate skips retrieval when it cannot help (illustrative proxy for Self-RAG Retrieve)')

  query                                                | gate: retrieve? | expected
  ------------------------------------------------------------------------------------
  When was Helios-7 launched?                          |            True | True
  Hello, how are you today?                            |           False | False
  What is 2+2?                                         |           False | False
  What is the ground resolution of the Helios-7 imager? |            True | True
  -> the gate skips retrieval when it cannot help (illustrative proxy for Self-RAG Retrieve)


## Step 7 — Self-RAG (b): the support check — ground before you trust (measured `ISSUP` proxy)

This one is genuinely computable: *does the answer's claim appear in the retrieved context?* Encode the
claim and each context sentence, take the **max cosine**. Score three claims of decreasing groundedness.

In [7]:
launch_res = retriever.retrieve('When was Helios-7 launched?', k=TOP_K)
ctx = retriever.parent_context(launch_res)
dense = retriever._dense
grounded = 'Helios-7 launched on March 3rd, 2024 from the Kourou spaceport.'   # true, in context
offtopic = 'Photosynthesis converts carbon dioxide and water into glucose.'    # unrelated -> reject
swap = 'Helios-7 launched in July 2021 from Cape Canaveral.'                    # same shape, WRONG facts
s_good = support_score(dense, grounded, ctx)
s_off = support_score(dense, offtopic, ctx)
s_swap = support_score(dense, swap, ctx)
print(f"  {'claim type':<34} | {'support':>7} | {'>= '+str(SUPPORT_THRESHOLD)+'?':>8}")
print('  ' + '-' * 58)
print(f"  {'grounded (true)':<34} | {s_good:>7.3f} | {is_supported(dense, grounded, ctx)!s:>8}")
print(f"  {'off-topic hallucination':<34} | {s_off:>7.3f} | {is_supported(dense, offtopic, ctx)!s:>8}")
print(f"  {'same-structure false (date swap)':<34} | {s_swap:>7.3f} | {is_supported(dense, swap, ctx)!s:>8}")
# what the proxy RELIABLY does: accept grounded, reject off-topic
assert s_good > s_off, 'grounded must score higher than off-topic'
assert is_supported(dense, grounded, ctx), 'grounded claim must clear the bar'
assert not is_supported(dense, offtopic, ctx), 'off-topic hallucination must FAIL the bar'
print('\n  -> ACCEPTS the grounded claim, REJECTS the off-topic hallucination.')

  claim type                         | support |  >= 0.5?
  ----------------------------------------------------------
  grounded (true)                    |   0.780 |     True
  off-topic hallucination            |   0.158 |    False


  same-structure false (date swap)   |   0.614 |     True



  -> ACCEPTS the grounded claim, REJECTS the off-topic hallucination.


**The honest limitation (measured).** The same-structure date-swap (`0.614`) *slips past* the 0.5 bar:
cosine measures **topical** similarity, not **factual entailment**. It still ranks below the true claim
(`0.780`), but a raw-cosine gate can't catch a plausible fact-swap — which is exactly why Self-RAG
*trains* an `ISSUP` token (an entailment judgement) rather than thresholding cosine.

In [8]:
assert s_swap > SUPPORT_THRESHOLD, 'the same-structure false claim slips past the cosine proxy (a real limit)'
assert s_good > s_swap, 'the true claim still scores higher than the same-structure false one'
print(f'same-structure false claim support = {s_swap:.3f}  (> {SUPPORT_THRESHOLD} -> slips past the cosine bar)')
print(f'true claim support = {s_good:.3f}  (still higher, but cosine != entailment)')
print('-> use a TRAINED critic / NLI for the support decision; cosine only catches off-topic drift.')

same-structure false claim support = 0.614  (> 0.5 -> slips past the cosine bar)
true claim support = 0.780  (still higher, but cosine != entailment)
-> use a TRAINED critic / NLI for the support decision; cosine only catches off-topic drift.


![Self-RAG support check (ISSUP proxy), measured: grounded 0.780 accept, off-topic 0.158 reject,
same-structure swap 0.614 slips past — the honest cosine-vs-entailment limit. Generated by
`make_figures_08.py`.](../../images/rag08_support_check.png)

![The Self-RAG reflect loop: Retrieve? gate → generate + ISREL/ISSUP/ISUSE critique → accept or
regenerate. Reflection-token definitions from Asai et al. 2023. Generated by
`make_figures_08.py`.](../../images/rag08_selfrag_loop.png)

## Step 8 — the library one-liners (parent-doc needs a vector store; Self-RAG needs a trained LM)

In production you don't hand-roll these. Parent-document retrieval is a few lines; Self-RAG needs a
*trained* model — shown for reference:

```python
# LangChain — ParentDocumentRetriever: retrieve small children, return larger parents.
from langchain.retrievers import ParentDocumentRetriever
from langchain_text_splitters import RecursiveCharacterTextSplitter
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore, docstore=store,
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=400),   # small, sharp embeddings
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=2000), # larger context returned
)

# LlamaIndex — AutoMergingRetriever: graded hierarchy (2048/512/128), merge leaves up to parents.
from llama_index.core.node_parser import HierarchicalNodeParser, get_leaf_nodes
from llama_index.core.retrievers import AutoMergingRetriever
nodes = HierarchicalNodeParser.from_defaults().get_nodes_from_documents(docs)  # chunk_sizes [2048,512,128]
# index the leaf nodes, keep all nodes in a docstore, then:
retriever = AutoMergingRetriever(base_index.as_retriever(similarity_top_k=6), storage_context)

# Self-RAG — needs the TRAINED reflection-token model (selfrag/selfrag_llama2_7b); the reflect loop
# is commonly wired with LangGraph (grade docs -> grade grounding -> grade usefulness -> regenerate).
```

---

### Recap

- **Parent-Document** decouples retrieval (small children) from generation (larger parents) — 2.1× context, same sharp retrieval.
- **RAG-Fusion** = multi-query + RRF (ch7) — recovers a fact the vague single query missed (recall 0→1).
- **Self-RAG** gates retrieval on demand and critiques grounding; the support-check works for off-topic drift but a plausible fact-swap needs a *trained* ISSUP.
- Full write-up: [the concept page](../08-Advanced-RAG-Parent-Doc-Fusion-Self-RAG.md).